# Форматы данных

__Авторы задач: Блохин Н.В. (NVBlokhin@fa.ru), Макрушин С.В. (SVMakrushin@fa.ru)__

Материалы:
* Макрушин С.В. "Форматы данных"
* https://docs.python.org/3/library/json.html
* https://docs.h5py.org/en/stable/
* https://www.crummy.com/software/BeautifulSoup/bs4/doc.ru/bs4ru.html
* Уэс Маккини. Python и анализ данных

## Задачи для совместного разбора

1. Вывести телефоны, содержащиеся в адресной книге `addres-book.json`

2. По данным из файла `addres-book-q.xml` сформировать список словарей с телефонами каждого из людей. 

3. Создайте 2 матрицы размера 1000x1000, используя различные параметризируемые распределения из numpy (https://docs.scipy.org/doc/numpy-1.15.0/reference/routines.random.html#distributions)

После этого сохраните получившиеся матрицы в hdf5-файл в виде двух различных датасетов. В качестве описания каждого датасета укажите параметры используемых распределений 

## Лабораторная работа №3

### JSON

1.1 Считайте файл `contributors_sample.json`, воспользовавшись модулем `json` и свяжите загруженные данные с переменной `contributors`. Выведите на экран уникальные почтовые домены, содержащиеся в почтовых адресах людей. Под доменом понимается часть адреса, следующая за символом `@`.

In [1]:
import json

In [2]:
with open('03_data_files_data/contributors_sample.json') as json_file:
    contributors = json.load(json_file)

In [3]:
contributors

[{'username': 'uhebert',
  'name': 'Lindsey Nguyen',
  'sex': 'F',
  'address': '01261 Cameron Spring\nTaylorfurt, AK 97791',
  'mail': 'jsalazar@gmail.com',
  'jobs': ['Energy engineer',
   'Engineer, site',
   'Environmental health practitioner',
   'Biomedical scientist',
   'Jewellery designer'],
  'id': 35193},
 {'username': 'vickitaylor',
  'name': 'Cheryl Lewis',
  'sex': 'F',
  'address': '66992 Welch Brooks\nMarshallshire, ID 56004',
  'mail': 'bhudson@gmail.com',
  'jobs': ['Music therapist',
   'Volunteer coordinator',
   'Designer, interior/spatial'],
  'id': 91970},
 {'username': 'sheilaadams',
  'name': 'Julia Allen',
  'sex': 'F',
  'address': 'Unit 1632 Box 2971\nDPO AE 23297',
  'mail': 'darren44@yahoo.com',
  'jobs': ['Management consultant',
   'Engineer, structural',
   'Lecturer, higher education',
   'Theatre manager',
   'Designer, textile'],
  'id': 1848091},
 {'username': 'nicole82',
  'name': 'Gina Stevens',
  'sex': 'F',
  'address': '9880 Michelle Bridge\nNe

In [4]:
domens = []
for contributor in contributors: 
    domens.append(contributor['mail'].split('@')[1])
domens = set(domens)
domens

{'gmail.com', 'hotmail.com', 'yahoo.com'}

1.2 Посчитайте, как часто встречается та или иная должность во всем наборе данных. Выведите на экран 5 должностей, которые встречаются наиболее часто. Для каждого пользователя из `contributors` выясните, какая из его должностей является наиболее распространенной (в смысле частоты упоминания во всем датасете) и добавьте ключ `top_job`, в котором хранится название этой должности.

In [5]:
jobs = {}
def get_top_job(contributor):
    for job in contributor['jobs']:
        if job in jobs:
            jobs[job] += 1
        else:
            jobs[job] = 1
    return max(jobs, key=jobs.get)

for contributor in contributors:
    contributor['top_job'] = get_top_job(contributor)

contributors

[{'username': 'uhebert',
  'name': 'Lindsey Nguyen',
  'sex': 'F',
  'address': '01261 Cameron Spring\nTaylorfurt, AK 97791',
  'mail': 'jsalazar@gmail.com',
  'jobs': ['Energy engineer',
   'Engineer, site',
   'Environmental health practitioner',
   'Biomedical scientist',
   'Jewellery designer'],
  'id': 35193,
  'top_job': 'Energy engineer'},
 {'username': 'vickitaylor',
  'name': 'Cheryl Lewis',
  'sex': 'F',
  'address': '66992 Welch Brooks\nMarshallshire, ID 56004',
  'mail': 'bhudson@gmail.com',
  'jobs': ['Music therapist',
   'Volunteer coordinator',
   'Designer, interior/spatial'],
  'id': 91970,
  'top_job': 'Energy engineer'},
 {'username': 'sheilaadams',
  'name': 'Julia Allen',
  'sex': 'F',
  'address': 'Unit 1632 Box 2971\nDPO AE 23297',
  'mail': 'darren44@yahoo.com',
  'jobs': ['Management consultant',
   'Engineer, structural',
   'Lecturer, higher education',
   'Theatre manager',
   'Designer, textile'],
  'id': 1848091,
  'top_job': 'Energy engineer'},
 {'usern

In [21]:
top_5_jobs = {}
for contributor in contributors:
    for job in contributor['jobs']:
        if job in top_5_jobs:
            top_5_jobs[job] += 1
        else:
            top_5_jobs[job] = 1

top_5_jobs = sorted(top_5_jobs.items(), key=lambda x: x[1], reverse=True)[:5]
top_5_jobs

[('Chemical engineer', 42),
 ('Chief Operating Officer', 41),
 ('Dance movement psychotherapist', 41),
 ('Bookseller', 41),
 ('Telecommunications researcher', 41)]

1.3 Создайте `pd.DataFrame` `contributors_df`, имеющий столбцы `id`, `username` и `sex` и `n_jobs`. Столбец `n_jobs` содержит кол-во должностей пользователя. При необходимости вы можете преобразовать исходные данные в списке `contributors` в удобный для создания `pd.DataFrame` вид.

Сгруппируйте полученные данные по столбцам `n_jobs` и `sex`. Выведите на экран серию `pd.Series`, в которой содержится информация о количестве человек в каждой группе.

In [9]:
import pandas as pd

In [10]:
contributors_df = pd.DataFrame(contributors)
n_jobs = contributors_df['jobs'].apply(len)
contributors_df['n_jobs'] = n_jobs

In [11]:
# оставить только столбцы id, username, sex, n_job в contributors_df
contributors_df.drop(['mail', 'jobs', 'top_job', 'name', 'address'], axis=1, inplace=True)
contributors_df


,username,sex,id,n_jobs
0,uhebert,F,35193,5
1,vickitaylor,F,91970,3
2,sheilaadams,F,1848091,5
3,nicole82,F,50969,3
4,jean67,M,676820,4
...,...,...,...,...
4195,stevenspencer,F,423555,4
4196,rwilliams,M,35251,5
4197,lmartinez,F,135887,5
4198,brendahill,M,212714,4


In [12]:
df = contributors_df.groupby('sex')['n_jobs'].value_counts()
df

sex  n_jobs
F    4         733
     3         703
     5         700
M    4         704
     3         690
     5         670
Name: n_jobs, dtype: int64

### XML

In [14]:
from bs4 import BeautifulSoup as bs

2.1 По данным файла `steps_sample.xml` сформируйте словарь с шагами по каждому рецепту вида `{id_рецепта: ["шаг1", "шаг2"]}`. Сохраните этот словарь в файл `steps_sample.json`. Выведите на экран шаги рецепта с id `84797`.

In [15]:
# make a dict with steps for each recipe {recipe_id: ["step1", "step2"]} from steps_sample.xml

In [16]:
with open('03_data_files_data/steps_sample.xml') as f:
    steps = bs(f, 'xml')

In [17]:
recipes_steps = {}

current_id = steps.find('id')
while current_id:
  recipes_steps[current_id.text] = [i.text for i in current_id.find_next('steps').find_all('step')]
  current_id = current_id.find_next('id')

In [18]:
recipes_steps

{'44123': ['in 1 / 4 cup butter , saute carrots , onion , celery and broccoli stems for 5 minutes',
  'add thyme , oregano and basil',
  'saute 5 minutes more',
  'add wine and deglaze pan',
  'add hot chicken stock and reduce by one-third',
  'add worcestershire sauce , tabasco , smoked chicken , beans and broccoli florets',
  'simmer 5 minutes',
  'add cream , simmer 5 minutes more and season to taste',
  'drop in remaining butter , piece by piece , stirring until melted and serve immediately',
  'smoked chicken: on a covered grill , slightly smoke boneless chicken , cooking to medium rare',
  'chef meskan uses applewood chips and does not allow the grill to become too hot'],
 '67664': ['mix all the ingredients using a blender',
  'pour into popsicle molds',
  'freeze and enjoy !'],
 '38798': ['combine all ingredients in a large bowl and mix well',
  'shape into one-inch balls',
  'cover and refrigerate or freeze until ready to bake',
  'preheat oven to 350 degrees',
  'place on ungr

In [19]:
with open('steps_sample.json', 'w', encoding='utf-8') as f:
    json.dump(recipes_steps, f, indent=2) # indent - отступы, json.dump - запись в файл


2.2 Получите список идентификаторов рецептов, в этапах выполнения которых есть информация о времени (часы или минуты). Для отбора подходящих рецептов обратите внимание на атрибуты соответствующих тэгов. Выведите на экран количество таких рецептов.

In [21]:
def get_recipes_with_time():
  curr = steps.find('id')
  while curr:
    steps_f = curr.find_next('steps')
    if steps_f.find('step', {'has_minutes': 1}) or steps_f.find('step', {'has_hours': 1}):
      yield curr.text 
    curr = curr.find_next('id')
s = [i for i in get_recipes_with_time()]
s

['44123',
 '35173',
 '453467',
 '306168',
 '50662',
 '118843',
 '149593',
 '200148',
 '310570',
 '95534',
 '109818',
 '66932',
 '226001',
 '141939',
 '250883',
 '120297',
 '223349',
 '60938',
 '302399',
 '342620',
 '296983',
 '129581',
 '325714',
 '487173',
 '447429',
 '137701',
 '63346',
 '342619',
 '383120',
 '463219',
 '39172',
 '216068',
 '173730',
 '287778',
 '437637',
 '123115',
 '371549',
 '376813',
 '390230',
 '401605',
 '306590',
 '299968',
 '192542',
 '147563',
 '193719',
 '38852',
 '250232',
 '437219',
 '77380',
 '21357',
 '198343',
 '129919',
 '375376',
 '63131',
 '24760',
 '375362',
 '217296',
 '435816',
 '198073',
 '202949',
 '367828',
 '140610',
 '392181',
 '468143',
 '176996',
 '290187',
 '459738',
 '111198',
 '33246',
 '302498',
 '165438',
 '267159',
 '401283',
 '428566',
 '533190',
 '478546',
 '40228',
 '255985',
 '402246',
 '180817',
 '133326',
 '513963',
 '213395',
 '482111',
 '292147',
 '235003',
 '449768',
 '257696',
 '53353',
 '115160',
 '140172',
 '392598',
 '35

In [22]:
len(s)

23469

2.3 Загрузите данные из файла `recipes_sample.csv` (__ЛР2__) в таблицу `recipes`. Для строк, которые содержат пропуски в столбце `n_steps`, заполните этот столбец на основе файла  `steps_sample.xml`. Строки, в которых столбец `n_steps` заполнен, оставьте без изменений. При решении задачи не используйте метод `iterrows` и аналогичные ему, позволяющие итерироваться по таблице построчно.

Проверьте, содержит ли столбец `n_steps` пропуски. Если нет, то преобразуйте его к целочисленному типу и сохраните результаты в файл `recipes_sample_with_filled_nsteps.csv`

In [29]:
recipes = pd.read_csv("02_pandas_data/recipes_sample.csv", parse_dates=['submitted'])
recipes

,name,id,minutes,contributor_id,submitted,n_steps,description,n_ingredients
0,george s at the cove black bean soup,44123,90,35193,2002-10-25,NaN,an original recipe created by chef scott meska...,18.0
1,healthy for them yogurt popsicles,67664,10,91970,2003-07-26,NaN,my children and their friends ask for my homem...,NaN
2,i can t believe it s spinach,38798,30,1533,2002-08-29,NaN,"these were so go, it surprised even me.",8.0
3,italian gut busters,35173,45,22724,2002-07-27,NaN,my sister-in-law made these for us at a family...,NaN
4,love is in the air beef fondue sauces,84797,25,4470,2004-02-23,4.0,i think a fondue is a very romantic casual din...,NaN
...,...,...,...,...,...,...,...,...
29995,zurie s holey rustic olive and cheddar bread,267661,80,200862,2007-11-25,16.0,this is based on a french recipe but i changed...,10.0
29996,zwetschgenkuchen bavarian plum cake,386977,240,177443,2009-08-24,NaN,"this is a traditional fresh plum cake, thought...",11.0
29997,zwiebelkuchen southwest german onion cake,103312,75,161745,2004-11-03,NaN,this is a traditional late summer early fall s...,NaN
29998,zydeco soup,486161,60,227978,2012-08-29,NaN,this is a delicious soup that i originally fou...,NaN


In [34]:
with open('03_data_files_data/steps_sample.xml') as f:
    steps = bs(f, 'xml')

In [38]:
recipes_len = {}

current_id = steps.find('id')
while current_id:
  steps_len = len(current_id.find_next('steps').find_all('step'))
  recipes_len[current_id.text] = steps_len 
  current_id = current_id.find_next('id')

recipes_len

{'44123': 11,
 '67664': 3,
 '38798': 5,
 '35173': 7,
 '84797': 4,
 '44045': 6,
 '107229': 8,
 '95926': 4,
 '453467': 12,
 '306168': 6,
 '50662': 15,
 '118843': 3,
 '69190': 5,
 '503475': 10,
 '149593': 10,
 '200148': 18,
 '310570': 38,
 '95534': 10,
 '109818': 7,
 '66932': 7,
 '226001': 12,
 '125195': 5,
 '141939': 13,
 '250883': 14,
 '120297': 14,
 '147477': 3,
 '223349': 7,
 '60938': 10,
 '302399': 11,
 '342620': 9,
 '296983': 14,
 '166089': 14,
 '129581': 33,
 '116741': 2,
 '325714': 6,
 '276594': 6,
 '487173': 30,
 '289671': 6,
 '44050': 2,
 '447429': 24,
 '137701': 18,
 '292568': 2,
 '299989': 14,
 '63346': 7,
 '342619': 9,
 '383120': 10,
 '367987': 3,
 '463219': 8,
 '39172': 8,
 '216068': 3,
 '173730': 28,
 '287778': 9,
 '437637': 10,
 '123115': 14,
 '371549': 8,
 '376813': 9,
 '134085': 4,
 '390230': 34,
 '401605': 7,
 '306590': 5,
 '303944': 13,
 '299968': 13,
 '192542': 4,
 '147563': 9,
 '193719': 16,
 '38852': 9,
 '250232': 10,
 '134787': 7,
 '437219': 9,
 '77380': 5,
 '21357

In [39]:
recipes['n_steps'] = recipes['n_steps'].fillna(recipes['id'].map(recipes_len))

In [40]:
recipes # хуй знает почему не заполнилось

,name,id,minutes,contributor_id,submitted,n_steps,description,n_ingredients
0,george s at the cove black bean soup,44123,90,35193,2002-10-25,NaN,an original recipe created by chef scott meska...,18.0
1,healthy for them yogurt popsicles,67664,10,91970,2003-07-26,NaN,my children and their friends ask for my homem...,NaN
2,i can t believe it s spinach,38798,30,1533,2002-08-29,NaN,"these were so go, it surprised even me.",8.0
3,italian gut busters,35173,45,22724,2002-07-27,NaN,my sister-in-law made these for us at a family...,NaN
4,love is in the air beef fondue sauces,84797,25,4470,2004-02-23,4.0,i think a fondue is a very romantic casual din...,NaN
...,...,...,...,...,...,...,...,...
29995,zurie s holey rustic olive and cheddar bread,267661,80,200862,2007-11-25,16.0,this is based on a french recipe but i changed...,10.0
29996,zwetschgenkuchen bavarian plum cake,386977,240,177443,2009-08-24,NaN,"this is a traditional fresh plum cake, thought...",11.0
29997,zwiebelkuchen southwest german onion cake,103312,75,161745,2004-11-03,NaN,this is a traditional late summer early fall s...,NaN
29998,zydeco soup,486161,60,227978,2012-08-29,NaN,this is a delicious soup that i originally fou...,NaN


### hdf

3.1 Выведите названия всех датасетов, находящихся в файле `nutrition_sample.h5`, а также размерность матриц, содержащихся в данных датасетах и их метаданные.

Формат вывода:
```
Dataset name=dataset_0 n_rows=30000 n_cols=2 col_0=recipe_id col_1=calories (#)
Dataset name=dataset_1 n_rows=30000 n_cols=2 col_0=recipe_id col_1=total fat (PDV)
Dataset name=dataset_2 n_rows=30000 n_cols=2 col_0=recipe_id col_1=sugar (PDV)
...
```

In [49]:
import h5py

In [59]:
datasets = {}
with h5py.File('03_data_files_data/nutrition_sample.h5', 'r') as h5file:
    for dataset in h5file:
        datasets[dataset] = h5file[dataset].shape
        meta = h5file[dataset].attrs

In [60]:
datasets

{'dataset_0': (30000, 2),
 'dataset_1': (30000, 2),
 'dataset_2': (30000, 2),
 'dataset_3': (30000, 2),
 'dataset_4': (30000, 2),
 'dataset_5': (30000, 2),
 'dataset_6': (30000, 2)}

In [62]:
for dataset in datasets:
    print(f'Dataset name={dataset} n_rows={datasets[dataset][0]} n_cols={datasets[dataset][1]}')

Dataset name=dataset_0 n_rows=30000 n_cols=2
Dataset name=dataset_1 n_rows=30000 n_cols=2
Dataset name=dataset_2 n_rows=30000 n_cols=2
Dataset name=dataset_3 n_rows=30000 n_cols=2
Dataset name=dataset_4 n_rows=30000 n_cols=2
Dataset name=dataset_5 n_rows=30000 n_cols=2
Dataset name=dataset_6 n_rows=30000 n_cols=2


3.2 Разбейте каждый из имеющихся датасетов на две части: 1 часть содержит только те строки, где PDV (Percent Daily Value) превышает 100%; 2 часть содержит те строки, где PDV составляет не более 100%. Создайте 2 группы в файле и разместите в них соответствующие части датасета c сохранением метаданных исходных датасетов. Итого должно получиться 2 группы, содержащие несколько датасетов. Датасеты, которые не содержат информацию о PDV, оставьте вне групп. Сохраните результаты в файл `nutrition_grouped.h5`.

3.3 Выведите названия всех групп и датасетов, находящихся в этих группах, из файла `nutrition_grouped.h5` а также размерность матриц, содержащихся в датасетах и их метаданные.

Формат вывода:
```
Dataset name=dataset_0 n_rows=30000 n_cols=2 col_0=recipe_id col_1=calories (#)
Group less_equal_than_100:
    Dataset name=less_equal_than_100/dataset_1 n_rows=28264 n_cols=2 col_0=recipe_id col_1=total fat (PDV)
    ....
Group more_than_100
    Dataset name=more_than_100/dataset_1 n_rows=1736 n_cols=2 col_0=recipe_id col_1=total fat (PDV)
    ....
....
```